# Polymarket Dashboard

Explore active events and markets on [Polymarket](https://polymarket.com) using the [Gamma API](https://docs.polymarket.com/market-data/fetching-markets).

| Section | Description |
|---------|-------------|
| 0 | Important tags (carousel / force-shown) |
| 1 | Top events by 24h volume |
| 2 | Top events by total volume |
| 3 | Top markets by 24h volume |
| 4 | Top markets by total volume |
| 5 | Biggest price movers (24h) |

In [ ]:
import requests
import pandas as pd
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

GAMMA_API = "https://gamma-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"

def fetch_events(order="volume24hr", limit=100):
    resp = requests.get(f"{GAMMA_API}/events", params={
        "active": "true",
        "closed": "false",
        "order": order,
        "ascending": "false",
        "limit": limit,
        "include_tag": "true",
    })
    resp.raise_for_status()
    return resp.json()

def fetch_markets(order="volume24hr", limit=100):
    resp = requests.get(f"{GAMMA_API}/markets", params={
        "active": "true",
        "closed": "false",
        "order": order,
        "ascending": "false",
        "limit": limit,
        "include_tag": "true",
    })
    resp.raise_for_status()
    return resp.json()

def events_df(events, n=10):
    rows = []
    for e in events[:n]:
        rows.append({
            "Title": e.get("title", "N/A"),
            "24h Volume": f"${float(e.get('volume24hr', 0)):,.0f}",
            "Total Volume": f"${float(e.get('volume', 0)):,.0f}",
            "Liquidity": f"${float(e.get('liquidity', 0)):,.0f}",
            "# Markets": len(e.get("markets", [])),
            "Slug": e.get("slug", ""),
        })
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    return df

def markets_df(markets, n=10):
    rows = []
    for m in markets[:n]:
        prices = m.get("outcomePrices", "[]") or "[]"
        rows.append({
            "Question": m.get("question", "N/A"),
            "24h Volume": f"${float(m.get('volume24hr', 0) or 0):,.0f}",
            "Total Volume": f"${float(m.get('volume', 0) or 0):,.0f}",
            "Liquidity": f"${float(m.get('liquidity', 0) or 0):,.0f}",
            "Outcome Prices": prices,
            "Slug": m.get("slug", ""),
        })
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    return df

def _get_tag_labels(item):
    return {t.get("label", "") for t in (item.get("tags") or [])}

def exclude_tags(items, tags):
    """Remove items that have any of the given tags."""
    tags = set(tags)
    return [i for i in items if not (_get_tag_labels(i) & tags)]

def keep_tags(items, tags):
    """Keep only items that have at least one of the given tags."""
    tags = set(tags)
    return [i for i in items if _get_tag_labels(i) & tags]

def get_yes_token_id(market):
    """Extract the Yes token ID (first entry) from clobTokenIds."""
    ids_str = market.get("clobTokenIds", "[]") or "[]"
    ids = json.loads(ids_str)
    return ids[0] if ids else None

def fetch_price_ago(token_id, interval="1d", timeout=10):
    """Fetch the earliest price point in the given interval from the CLOB API.
    interval: '1h', '6h', '1d', '1w', '1m'
    """
    fidelity = {"1h": 1, "6h": 5, "1d": 60, "1w": 360, "1m": 1440}.get(interval, 60)
    try:
        resp = requests.get(f"{CLOB_API}/prices-history", params={
            "market": token_id,
            "interval": interval,
            "fidelity": fidelity,
        }, timeout=timeout)
        resp.raise_for_status()
        history = resp.json().get("history", [])
        if history:
            return history[0]["p"]
    except Exception:
        pass
    return None

print("Helpers loaded.")

---
## Section 1: Top Events by 24h Volume

In [ ]:
events_by_24h = fetch_events(order="volume24hr")
print(f"Fetched {len(events_by_24h)} events (ordered by 24h volume)")
events_df(events_by_24h)

---
## Section 2: Top Events by Total Volume

In [ ]:
events_by_total = fetch_events(order="volume")
print(f"Fetched {len(events_by_total)} events (ordered by total volume)")
events_df(events_by_total)

---
## Section 3: Top Markets by 24h Volume

In [ ]:
markets_by_24h = fetch_markets(order="volume24hr")
print(f"Fetched {len(markets_by_24h)} markets (ordered by 24h volume)")
markets_df(markets_by_24h)

In [ ]:
print(json.dumps(events_by_24h[4], indent=2))

In [ ]:
print(json.dumps(markets_by_24h[2], indent=2))

---
## Section 4: Top Markets by Total Volume

In [ ]:
markets_by_total = fetch_markets(order="volumeNum")
print(f"Fetched {len(markets_by_total)} markets (ordered by total volume)")
markets_df(markets_by_total)

---
## Section 5: Biggest Price Movers

Use the dropdown to pick a lookback window (1 hour → 1 month). Fetches actual price history from the CLOB `/prices-history` endpoint and computes `current_price − price_then` in percentage points.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

PERIODS = {"1 Hour": "1h", "6 Hours": "6h", "1 Day": "1d", "1 Week": "1w", "1 Month": "1m"}

all_markets = fetch_markets(order="volume24hr", limit=1000)
filtered = exclude_tags(all_markets, ["Crypto Prices", "Sports"])
candidates = [m for m in filtered if float(m.get("volume24hr", 0) or 0) > 0]
print(f"Loaded {len(candidates)} candidate markets (from {len(all_markets)} total)")

period_picker = widgets.Dropdown(
    options=list(PERIODS.keys()),
    value="1 Day",
    description="Lookback:",
    style={"description_width": "auto"},
)
out = widgets.Output()

def refresh(_=None):
    with out:
        clear_output(wait=True)
        interval = PERIODS[period_picker.value]
        print(f"Fetching {period_picker.value} price history for {len(candidates)} markets...")

        def _compute(m):
            token_id = get_yes_token_id(m)
            if not token_id:
                return None
            price_then = fetch_price_ago(token_id, interval=interval)
            if price_then is None:
                return None
            prices_str = m.get("outcomePrices", "[]") or "[]"
            prices = json.loads(prices_str)
            if not prices:
                return None
            current = float(prices[0])
            return {**m, "_change": current - price_then,
                    "_abs_change": abs(current - price_then),
                    "_current_yes": current}

        results = []
        with ThreadPoolExecutor(max_workers=20) as pool:
            for r in pool.map(_compute, candidates):
                if r is not None:
                    results.append(r)

        movers = sorted(results, key=lambda m: m["_abs_change"], reverse=True)

        rows = []
        for m in movers[:20]:
            rows.append({
                "Question": m.get("question", "N/A"),
                "Change": f"{m['_change']:+.0%}",
                "Current": f"{m['_current_yes']:.0%}",
                "24h Volume": f"${float(m.get('volume24hr', 0) or 0):,.0f}",
                "Outcome Prices": m.get("outcomePrices", "[]") or "[]",
                "Slug": m.get("slug", ""),
            })

        df = pd.DataFrame(rows)
        df.index = range(1, len(df) + 1)
        df.index.name = "Rank"
        print(f"Got history for {len(results)}/{len(candidates)} markets\n")
        print(f"Top 20 movers — {period_picker.value} window — out of {len(all_markets)} active markets")
        display(df)

period_picker.observe(refresh, names="value")
display(period_picker, out)
refresh()